In [1]:
import numpy as np
from subprocess import PIPE, run
import matplotlib.pyplot as plt
import os
import textwrap
from waxx.control import ethernet_relay

class ExptBuilder():
    def __init__(self):
        self.__code_path__ = os.environ.get('code')
        self.__temp_exp_path__ = os.path.join(self.__code_path__, "k-exp", "kexp", "experiments", "ml_expt.py")

    def run_expt(self):
        expt_path = self.__temp_exp_path__
        run_expt_command = r"%kpy% & ar " + expt_path
        result = run(run_expt_command, stdout=PIPE, stderr=PIPE, universal_newlines=True, shell=True)
        print(result.returncode, result.stdout, result.stderr)
        os.remove(self.__temp_exp_path__)
        return result.returncode
    
    def write_experiment_to_file(self, program):
        with open(self.__temp_exp_path__, 'w') as file:
            file.write(program)
    
    def fringe_scan_expt(self, img_amp, freq_raman):
        script = textwrap.dedent(f"""
        from artiq.experiment import *
        from artiq.experiment import delay
        from kexp import Base, img_types, cameras
        import numpy as np
        from kexp.calibrations.tweezer import tweezer_vpd1_to_vpd2
        from kexp.calibrations.imaging import high_field_imaging_detuning
        from artiq.coredevice.sampler import Sampler
        from artiq.language import now_mu
        from kexp.util.artiq.async_print import aprint

        class hf_monitored_rabi(EnvExperiment, Base):

            def prepare(self):
                Base.__init__(self,setup_camera=True,
                            camera_select=cameras.andor,
                            save_data=True,
                            imaging_type=img_types.DISPERSIVE)

                self.p.v_pd_hf_tweezer_squeeze_power = 3.94
                                 
                self.p.frequency_raman_transition = {freq_raman}

                self.p.t_continuous_rabi = 450.e-6
                
                self.p.amp_imaging = {img_amp}
                
                self.p.dimension_slm_mask = 20.e-6

                self.p.phase_slm_mask = 0.2 * np.pi

                self.p.t_tweezer_hold = 20.e-3

                self.p.t_tof = 20.e-6
                
                self.p.t_mot_load = 1.0
                
                self.p.N_repeats = 10

                self.scope = self.scope_data.add_siglent_scope("192.168.1.108", label='PD', arm=False)

                self.finish_prepare(shuffle=True)

            @kernel
            def scan_kernel(self):
                
                self.set_imaging_detuning(frequency_detuned = self.p.frequency_detuned_hf_midpoint)
                self.slm.write_phase_mask_kernel(phase=self.p.phase_slm_mask,dimension=self.p.dimension_slm_mask)
                self.imaging.set_power(self.p.amp_imaging)

                self.prepare_hf_tweezers(squeeze=True)

                self.raman.init(fraction_power = self.p.fraction_power_raman,
                                frequency_transition = self.p.frequency_raman_transition)

                self.ttl.raman_shutter.on()
                delay(10.e-3)
                self.ttl.line_trigger.wait_for_line_trigger()
                delay(4.7e-3)
                
                self.ttl.pd_scope_trig3.pulse(1.e-6)
                self.imaging.on()
                delay(3.e-6)

                self.raman.pulse(t=self.p.t_continuous_rabi)
                
                self.imaging.off()

                self.ttl.raman_shutter.off()
                
                self.set_imaging_detuning(frequency_detuned = self.p.frequency_detuned_hf_f1m1)
                self.imaging.set_power(.2,reset_pid=True)

                delay(self.p.t_tweezer_hold)
                self.tweezer.off()

                delay(self.p.t_tof)

                self.abs_image()

                self.core.wait_until_mu(now_mu())
                self.scope.read_sweep(0)
                self.core.break_realtime()
                delay(30.e-3)

            @kernel
            def run(self):
                self.init_kernel()
                self.load_2D_mot(self.p.t_2D_mot_load_delay)
                self.scan()
                self.mot_observe()

            def analyze(self):
                import os
                expt_filepath = os.path.abspath(__file__)
                # aprint(self.scope._data)
                self.end(expt_filepath)
                """)
        return script

In [2]:
eBuilder = ExptBuilder()

In [3]:
img_amp = np.linspace(.2,2.5,100)

freq_raman = np.array([1.19485943e+08, 1.19488412e+08, 1.19490881e+08, 1.19493351e+08,
       1.19495820e+08, 1.19498289e+08, 1.19500758e+08, 1.19503228e+08,
       1.19505697e+08, 1.19508166e+08, 1.19510635e+08, 1.19513105e+08,
       1.19515574e+08, 1.19518043e+08, 1.19520513e+08, 1.19522982e+08,
       1.19525451e+08, 1.19527920e+08, 1.19530390e+08, 1.19532859e+08,
       1.19535328e+08, 1.19537797e+08, 1.19540267e+08, 1.19542736e+08,
       1.19545205e+08, 1.19547674e+08, 1.19550144e+08, 1.19552613e+08,
       1.19555082e+08, 1.19557552e+08, 1.19560021e+08, 1.19562490e+08,
       1.19564959e+08, 1.19567429e+08, 1.19569898e+08, 1.19572367e+08,
       1.19574836e+08, 1.19577306e+08, 1.19579775e+08, 1.19582244e+08,
       1.19584713e+08, 1.19587183e+08, 1.19589652e+08, 1.19592121e+08,
       1.19594590e+08, 1.19597060e+08, 1.19599529e+08, 1.19601998e+08,
       1.19604468e+08, 1.19606937e+08, 1.19609406e+08, 1.19611875e+08,
       1.19614345e+08, 1.19616814e+08, 1.19619283e+08, 1.19621752e+08,
       1.19624222e+08, 1.19626691e+08, 1.19629160e+08, 1.19631629e+08,
       1.19634099e+08, 1.19636568e+08, 1.19639037e+08, 1.19641506e+08,
       1.19643976e+08, 1.19646445e+08, 1.19648914e+08, 1.19651384e+08,
       1.19653853e+08, 1.19656322e+08, 1.19658791e+08, 1.19661261e+08,
       1.19663730e+08, 1.19666199e+08, 1.19668668e+08, 1.19671138e+08,
       1.19673607e+08, 1.19676076e+08, 1.19678545e+08, 1.19681015e+08,
       1.19683484e+08, 1.19685953e+08, 1.19688423e+08, 1.19690892e+08,
       1.19693361e+08, 1.19695830e+08, 1.19698300e+08, 1.19700769e+08,
       1.19703238e+08, 1.19705707e+08, 1.19708177e+08, 1.19710646e+08,
       1.19713115e+08, 1.19715584e+08, 1.19718054e+08, 1.19720523e+08,
       1.19722992e+08, 1.19725461e+08, 1.19727931e+08, 1.19730400e+08])


for f in range(len(img_amp)):
    print(img_amp[f],freq_raman[f])
    eBuilder.write_experiment_to_file(eBuilder.fringe_scan_expt(img_amp=img_amp[f],freq_raman = freq_raman[f]))
    eBuilder.run_expt()

0.2 119485943.0
0 Failed to connect to wavemeter -- please close wavemeter software if it is open. Lock status will not be checked.
 10 values of dummy. 10 total shots. 30 total images expected.
Run ID: 69040
Acknowledged camera ready signal.
Camera is ready.

Sent: {'mask': 'spot', 'center': [994, 820], 'phase': 0.2, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.2 pi, x-center = 994, y-center = 820

 Run ID: 69040

Sent: {'mask': 'spot', 'center': [994, 820], 'phase': 0.2, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.2 pi, x-center = 994, y-center = 820

shot 1/10 done

Sent: {'mask': 'spot', 'center': [994, 820], 'phase': 0.2, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.2 pi, x-center = 994, y-center = 820

shot 2/10 done

Sent: {'mask': 'spot', 'center': [994, 820], 'phase': 0.2, 'dimension'